# India Runs — Track 1 Candidate Ranking — Sandbox

**Team repo:** https://github.com/Lucky-1608/Resume-analyzer-and-ranker

This notebook is the mandatory sandbox (Section 10.5). It:
1. Clones the team's GitHub repository (the exact code that produced the full submission)
2. Installs the dependencies from `requirements.txt`
3. Runs the **same ranking pipeline** on a pre-loaded 100-candidate sample
4. Validates the output against the official format rules
5. Displays the ranked CSV

Everything runs **CPU-only, no network during ranking, no GPU, no hosted LLM** — matching the contest compute constraints. On a 100-candidate sample it completes in **a few seconds** (the full 100K pool runs in ~150s, well under the 5-minute limit).

**To run:** Runtime → Run all (or press the play button on each cell top to bottom).

## Step 1 — Clone the repository

In [ ]:
!rm -rf Resume-analyzer-and-ranker
!git clone https://github.com/Lucky-1608/Resume-analyzer-and-ranker.git
%cd Resume-analyzer-and-ranker

## Step 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## Step 3 — Run the ranking pipeline on the 100-candidate sample

Uses the exact same entry point and code path as the full submission — only the input file is the small sample bundled in `sandbox/`.

In [ ]:
import time
t0 = time.time()
!python run_submission.py \
    --candidates sandbox/sample_candidates.jsonl \
    --jd sandbox/job_description.docx \
    --out sandbox_submission.csv
print(f'\nWall-clock time: {time.time()-t0:.1f}s (limit: 300s)')

## Step 4 — Validate the output against the official format rules

In [ ]:
import csv, re
rows = list(csv.DictReader(open('sandbox_submission.csv', encoding='utf-8')))
ids = [r['candidate_id'] for r in rows]
ranks = [int(r['rank']) for r in rows]
scores = [float(r['score']) for r in rows]
pat = re.compile(r'^CAND_[0-9]{7}$')
n = len(rows)

checks = {
    f'Row count = {n}': n == 100,
    'All candidate_ids match CAND_XXXXXXX': all(pat.match(i) for i in ids),
    'All candidate_ids unique': len(set(ids)) == n,
    f'Ranks are 1..{n} exactly once': sorted(ranks) == list(range(1, n+1)),
    'Scores non-increasing with rank': all(scores[i] >= scores[i+1] for i in range(n-1)),
    'Header correct': list(rows[0].keys()) == ['candidate_id','rank','score','reasoning'],
    'Reasoning non-empty for all rows': all(r['reasoning'].strip() for r in rows),
    'Reasoning all unique': len({r['reasoning'] for r in rows}) == n,
}
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
print()
print('ALL CHECKS PASSED ✅' if all(checks.values()) else 'SOME CHECKS FAILED ❌')

## Step 5 — Show the ranked output

In [ ]:
import pandas as pd
df = pd.read_csv('sandbox_submission.csv')
print(f'Ranked {len(df)} candidates. Top 10:')
for _, r in df.head(10).iterrows():
    print(f"\n  #{r['rank']}  {r['candidate_id']}  (score={r['score']})")
    print(f"     {r['reasoning']}")
df

---
### Notes for reviewers

- This sandbox runs the **identical pipeline** used for the full 100K submission — same `run_submission.py`, same `src/` modules. Only the input is the 100-candidate sample.
- The ranking is **deterministic**: re-running produces a byte-identical CSV (verified on the full pool across multiple seeds).
- **No network calls, no GPU, no hosted LLM** during ranking — pure scikit-learn + python-docx.
- Full architecture (RRF fusion, two-stage honeypot detection, evidence coherence, 23-signal behavioral model) is documented in the repo `README.md`.
- To reproduce the **full** submission, run on the complete `candidates.jsonl`:
  `python run_submission.py --candidates data/candidates.jsonl --jd data/job_description.docx --out submission.csv`